In [ ]:
# ========================
# IA Multi-output: prever CONVERSAO + FATOR juntos
# ========================

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

# ---------- 1) Ler base ----------
tabela = pd.read_excel("Banco_de_Fardos.xlsx")
tabela.columns = tabela.columns.str.strip()
tabela = tabela[["MATERIAL", "NOME_CONCO", "CONVERSAO", "FATOR"]].copy()

# ---------- 2) Preprocess ----------
tabela["FATOR"] = pd.to_numeric(tabela["FATOR"], errors="coerce").fillna(1.0)

# Encoders
enc_m = LabelEncoder(); tabela["MATERIAL_enc"] = enc_m.fit_transform(tabela["MATERIAL"].astype(str))
enc_n = LabelEncoder(); tabela["NOME_enc"]     = enc_n.fit_transform(tabela["NOME_CONCO"].astype(str))
enc_conv = LabelEncoder(); tabela["CONV_enc"] = enc_conv.fit_transform(tabela["CONVERSAO"].astype(str))

# Features e target conjunto
X = tabela[["MATERIAL_enc", "NOME_enc"]]
y_multi = np.column_stack([tabela["FATOR"].values, tabela["CONV_enc"].values])  # (n,2)

# ---------- 3) Treinar ----------
modelo_multi = RandomForestRegressor(random_state=42, n_estimators=150)
modelo_multi.fit(X, y_multi)

print("Treinamento multi-output concluído.")

# ---------- 4) Previsão ----------
novos = pd.read_excel("novos_fardos.xlsx")
novos.columns = novos.columns.str.strip()
novos = novos[["MATERIAL", "NOME_CONCO"]].copy()

def safe_map(series, encoder):
    m = {v:i for i,v in enumerate(encoder.classes_)}
    return series.astype(str).map(m).fillna(-1).astype(int)

novos["MATERIAL_enc"] = safe_map(novos["MATERIAL"], enc_m)
novos["NOME_enc"] = safe_map(novos["NOME_CONCO"], enc_n)

X_novos = novos[["MATERIAL_enc", "NOME_enc"]]

# Previsão conjunta
pred_multi = modelo_multi.predict(X_novos)

pred_fator = pred_multi[:,0]
pred_conv_enc = np.rint(pred_multi[:,1]).astype(int)
pred_conv_enc = np.clip(pred_conv_enc, 0, len(enc_conv.classes_)-1)
pred_conv = enc_conv.inverse_transform(pred_conv_enc)

# Resultado final
resultado = novos.copy()
resultado["CONVERSAO"] = pred_conv
resultado["FATOR"] = pred_fator

resultado.to_excel("PlanilhaAtualizada_multioutput.xlsx", index=False)
print("Resultado salvo: PlanilhaAtualizada_multioutput.xlsx")
